In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develope a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-05-29
Last Modified: 2026-05-29
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""--------------------------------------------"""
"""THURSDAY"""
# change the tent fitting to use the all trials fit baseline
# add movement (average normalized me on a trial)

# unit and sanity checks!!!
# ---------*
# # cases
# 1. regular, no filtering
# 2. only mb or mf trials (balance and no balance)
# 3. both mb and mf trials, balanced
# 4. use passed idx_subsamps
# ---------*

"""FRIDAY"""
# one regressor
# add time
"""--------------------------------------------"""

## both

In [ ]:
"""TODO: inform michael"""
# no outlier trial filtering at the moment
# for session subsampling, i divided the number of tents so that the frequency of slow drift is the same

In [ ]:
from sg.models import Encoder

encoder = Encoder(subj_id, sess_id, num_tents=12)
encoder.fit_encoder()
encoder.encoder_predict()

In [ ]:
encoder.verify(subtract_baseline=False)

In [ ]:
from squiggs.renderers import FitRenderer
from squiggs.neuron_viewer import NeuronViewer
from utils.paths import FIGURES_DIR

reg = "DLS"

r = FitRenderer(
    y=encoder.robs[:, encoder.reg_idxs[reg]],
    yhat=encoder.robs_predict["encoder"][:, encoder.reg_idxs[reg]],
    mode="lite",
)

_ = NeuronViewer(num_units=encoder.num_units, render_func=r, fig_dir=FIGURES_DIR)

In [ ]:
from squiggs.renderers import PETHWeightRenderer
from squiggs.neuron_viewer import NeuronViewer
from utils.paths import FIGURES_DIR
from core.data import get_psths_cond, get_choice_ts, get_tavg_sc_cond

"""
drift: 21, 29, 35, 36
response: 16, 25, 26, 28
"""

reg = "DLS"
mode = "response"

sc_tavg = get_tavg_sc_cond(
    encoder.robs[:, encoder.reg_idxs[reg]], encoder.trial_data, cond=mode
)

r = PETHWeightRenderer(
    weights=encoder.encoder.coef_[encoder.reg_idxs[reg], :],
    weight_names=encoder.dm_names,
    robs=encoder.robs[:, encoder.reg_idxs[reg]],
    sc_tavg=sc_tavg,
    event_times=get_choice_ts(encoder.trial_data, mode=mode),
    spike_times=encoder.spike_times[reg],
    peths=get_psths_cond(encoder.psths[reg], encoder.trial_data, mode=mode),
    pres=0.5,
    posts=1,
    binwidth_s=25 / 1000,
)

nv = NeuronViewer(num_units=len(encoder.psths[reg]), render_func=r, fig_dir=FIGURES_DIR)

In [ ]:
from sg.models import ShuffledEncoder

se = ShuffledEncoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "block_side",
        "strategy",
        "response_prev",
        "rewarded_prev",
    ],
)
se.plot_cvr2()
se.plot_dr2()

# weight distro across sessions

In [ ]:
import numpy as np
from core.data import subject_ids, session_ids

sess_ids = session_ids[np.where(subject_ids == subj_id)[0][0]]
coefs_sess = {"all": [], "DLS": [], "DMS": []}
regressors = np.array(
    ["response_left", "response_right", "rewarded_incorr", "rewarded_corr"]
)

for sess_id in sess_ids:
    encoder = Encoder(subj_id, sess_id, num_tents=5)
    encoder.fit_encoder()

    # get coefs and separate by region as well
    coefs = encoder.encoder.coef_

    tv_idxs = [i for i, dm_name in enumerate(encoder.dm_names) if dm_name in regressors]

    coefs_ = coefs[:, tv_idxs]

    coefs_sess["all"].append(coefs_)
    coefs_sess["DLS"].append(coefs_[encoder.reg_idxs["DLS"]])
    coefs_sess["DMS"].append(coefs_[encoder.reg_idxs["DMS"]])

In [ ]:
regr = "response"
val = "left"


def plot_coefs_boxplot(coefs_sess, sess_ids, reg="all", regr="response", val="left"):
    regr_idx = np.where(regressors == f"{regr}_{val}")
    coefs_regr = [
        np.squeeze(coefs_sess_[:, regr_idx]) for coefs_sess_ in coefs_sess[reg]
    ]

    fig, ax = plt.subplots(figsize=(6, 3), tight_layout=True)

    ax.boxplot(
        coefs_regr,
        widths=0.5,
        patch_artist=True,
        flierprops=dict(
            marker="o",
            markersize=1.5,
            alpha=0.4,
            markeredgewidth=0,
            markerfacecolor="steelblue",
        ),
        medianprops=dict(color="k", linewidth=1),
        boxprops=dict(facecolor="steelblue", alpha=0.6),
        whiskerprops=dict(linewidth=0.8),
        capprops=dict(linewidth=0.8),
    )

    ax.axhline(y=0, color="k", linewidth=0.5, linestyle="--", zorder=0)

    ax.set_xlabel("Session")
    ax.set_ylabel(rf"$\beta_{{\mathrm{{{regr}\_{val}}}}}$")
    ax.set_xticks(np.arange(len(sess_ids)) + 1)
    ax.set_xticklabels(sess_ids, rotation=45, ha="right")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.show()


plot_coefs_boxplot(coefs_sess, sess_ids, reg="all", regr="response", val="left")

In [ ]:
import pandas as pd
import seaborn as sns


def coefs_to_dict(coefs_sess):
    dfs = []
    for i, sess_id in enumerate(sess_ids):
        df = pd.DataFrame(coefs_sess["all"][i], columns=regressors)
        df.insert(0, "key", sess_id)
        dfs.append(df)

    coefs_df = pd.concat(dfs, ignore_index=True)
    return coefs_df


def plot_coefs_ridge(coefs_sess, sess_ids, regr="response", val="left"):
    saved_params = plt.rcParams.copy()

    sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

    df = coefs_to_dict(coefs_sess)
    regr_key = f"{regr}_{val}"

    """from sns gallery"""
    # Initialize the FacetGrid object
    pal = sns.cubehelix_palette(len(sess_ids), rot=-0.25, light=0.7)
    g = sns.FacetGrid(df, row="key", hue="key", aspect=15, height=0.5, palette=pal)

    # Draw the densities in a few steps
    g.map(
        sns.kdeplot,
        regr_key,
        bw_adjust=0.5,
        clip_on=False,
        fill=True,
        alpha=1,
        linewidth=1.5,
    )
    g.map(sns.kdeplot, regr_key, clip_on=False, color="w", lw=2, bw_adjust=0.5)

    # passing color=None to refline() uses the hue mapping
    g.refline(y=0, linewidth=2, linestyle="-", color=None, clip_on=False)

    # Define and use a simple function to label the plot in axes coordinates
    def label(x, color, label):
        ax = plt.gca()
        ax.text(
            0,
            0.2,
            label,
            fontweight="bold",
            color=color,
            ha="left",
            va="center",
            transform=ax.transAxes,
        )

    g.map(label, regr_key)

    # Set the subplots to overlap
    g.figure.subplots_adjust(hspace=-0.25)

    # Remove axes details that don't play well with overlap
    g.set_titles("")
    g.set(yticks=[], ylabel="")
    g.despine(bottom=True, left=True)

    plt.rcParams.update(saved_params)

In [ ]:
plot_coefs_ridge(coefs_sess, sess_ids, regr="rewarded", val="corr")

# strategy split

## single session

In [ ]:
encoder = Encoder(subj_id, sess_id, num_tents=5)
encoder.fit_encoder()
encoder.encoder_predict()
encoder.verify(subtract_baseline=True)

In [ ]:
encoder_mb = Encoder(subj_id, sess_id, strategy_filter="mb", num_tents=12)
encoder_mb.fit_encoder()

encoder_mf = Encoder(subj_id, sess_id, strategy_filter="mf", num_tents=12)
encoder_mf.fit_encoder()

In [ ]:
encoder.verify(subtract_baseline=True)

In [ ]:
encoder_mb.verify(subtract_baseline=False)

In [ ]:
from squiggs.renderers import PETHWeightRenderer

reg = "DLS"
mode = "response"

sc_tavg = get_tavg_sc_cond(
    encoder_mb.robs[:, encoder_mb.reg_idxs[reg]], encoder_mb.trial_data, cond=mode
)

r = PETHWeightRenderer(
    weights=encoder_mb.encoder.coef_[encoder.reg_idxs[reg], :],
    weight_names=encoder_mb.dm_names,
    robs=encoder_mb.robs[:, encoder_mb.reg_idxs[reg]],
    sc_tavg=sc_tavg,
    event_times=get_choice_ts(encoder_mb.trial_data, mode=mode),
    spike_times=encoder_mb.spike_times[reg],
    peths=get_psths_cond(encoder_mb.psths[reg], encoder_mb.trial_data, mode=mode),
    pres=0.5,
    posts=1,
    binwidth_s=25 / 1000,
)

nv = NeuronViewer(
    num_units=len(encoder_mb.psths[reg]), render_func=r, fig_dir=FIGURES_DIR
)

In [ ]:
encoder_mf.verify()

In [ ]:
from squiggs.renderers import PETHWeightRenderer

reg = "DLS"
mode = "response"

sc_tavg = get_tavg_sc_cond(
    encoder_mf.robs[:, encoder_mf.reg_idxs[reg]], encoder_mf.trial_data, cond=mode
)

r = PETHWeightRenderer(
    weights=encoder_mf.encoder.coef_[encoder_mf.reg_idxs[reg], :],
    weight_names=encoder_mf.dm_names,
    robs=encoder_mf.robs[:, encoder_mf.reg_idxs[reg]],
    sc_tavg=sc_tavg,
    event_times=get_choice_ts(encoder_mf.trial_data, mode=mode),
    spike_times=encoder_mf.spike_times[reg],
    peths=get_psths_cond(encoder_mf.psths[reg], encoder_mf.trial_data, mode=mode),
    pres=0.5,
    posts=1,
    binwidth_s=25 / 1000,
)

nv = NeuronViewer(
    num_units=len(encoder_mf.psths[reg]), render_func=r, fig_dir=FIGURES_DIR
)

In [ ]:
# weight comparison
import numpy as np

scale = 10

regr = "response"
val = "left"

i = np.where(encoder.dm_names == f"{regr}_{val}")[0][0]

plt.figure(tight_layout=True)
plt.scatter(
    encoder_mb.encoder.coef_[:, i], encoder_mf.encoder.coef_[:, i], s=0.5, alpha=0.5
)

plt.plot([-1, 1], [-1, 1], color="#666666", linewidth=0.5, linestyle="--")
# plt.plot([-1/scale, 1/scale], [-1, 1], color="#173094", linewidth=0.5, linestyle='--', label=f"{scale}*mb") # anything to the left/top of this line means that mf has double the magnitude of mb
# plt.plot([-1, 1], [-1/scale, 1/scale], color="#CCA415", linewidth=0.5, linestyle='--', label=f"{scale}*mf") # anything to the right/bottom of this line means that mb has double the magnitude of mf
# # anything in the hourglass is considered to be close enough to the unity line

plt.axhline(y=0, color="k", linewidth=0.5)
plt.axvline(x=0, color="k", linewidth=0.5)
plt.xlabel(f"mb, bweight {regr} {val}")
plt.ylabel(f"mf, bweight {regr} {val}")
plt.legend()
plt.plot()

In [ ]:
regr = "response"
val = "left"
regr_idx = np.where(encoder.dm_names == f"{regr}_{val}")[0][0]

scale = 10

coef_mb = encoder_mb.encoder.coef_[:, regr_idx]
coef_mf = encoder_mf.encoder.coef_[:, regr_idx]

mb_gr_idxs = np.where(np.abs(coef_mb) > np.abs(coef_mf) * scale)[0]
mf_gr_idxs = np.where(np.abs(coef_mf) > np.abs(coef_mb) * scale)[0]

In [ ]:
mb_gr_idxs

In [ ]:
from squiggs.renderers import PETHWeightCompRenderer
from squiggs.neuron_viewer import NeuronViewer
from core.data import get_tavg_sc_cond, get_choice_ts, get_psths_cond
from utils.paths import FIGURES_DIR

reg = "DLS"
mode = "response"

sc_tavg_mb = get_tavg_sc_cond(
    encoder_mb.robs[:, encoder_mb.reg_idxs[reg]], encoder_mb.trial_data, cond=mode
)

sc_tavg_mf = get_tavg_sc_cond(
    encoder_mf.robs[:, encoder_mf.reg_idxs[reg]], encoder_mf.trial_data, cond=mode
)

r = PETHWeightCompRenderer(
    weights={
        "mb": encoder_mb.encoder.coef_[encoder_mb.reg_idxs[reg], :],
        "mf": encoder_mf.encoder.coef_[encoder_mf.reg_idxs[reg], :],
    },
    weight_names=encoder_mb.dm_names,
    robs={
        "mb": encoder_mb.robs[:, encoder_mb.reg_idxs[reg]],
        "mf": encoder_mf.robs[:, encoder_mf.reg_idxs[reg]],
    },
    sc_tavgs={"mb": sc_tavg_mb, "mf": sc_tavg_mf},
    event_times={
        "mb": get_choice_ts(encoder_mb.trial_data, mode=mode),
        "mf": get_choice_ts(encoder_mf.trial_data, mode=mode),
    },
    spike_times=encoder_mb.spike_times[reg],  # == encoder_mf.spike_times
    peths={
        "mb": get_psths_cond(encoder_mb.psths[reg], encoder_mb.trial_data, mode=mode),
        "mf": get_psths_cond(encoder_mf.psths[reg], encoder_mf.trial_data, mode=mode),
    },
)

_ = NeuronViewer(encoder_mb.num_units, r, fig_dir=FIGURES_DIR)

## aggregate

In [ ]:
import numpy as np
from core.data import subject_ids, session_ids

sess_ids = session_ids[np.where(subject_ids == subj_id)[0][0]]
coefs = {"mb": {"DLS": [], "DMS": []}, "mf": {"DLS": [], "DMS": []}}
regressors = np.array(
    ["response_left", "response_right", "rewarded_incorr", "rewarded_corr"]
)

for sess_id in sess_ids:
    encoder_mb = Encoder(subj_id, sess_id, strategy_filter="mb", num_tents=12)
    encoder_mf = Encoder(subj_id, sess_id, strategy_filter="mf", num_tents=12)

    try:
        encoder_mb.fit_encoder()
        encoder_mf.fit_encoder()
    except RuntimeError:
        continue

    # get coefs and separate by region as well
    coefs_mb = encoder_mb.encoder.coef_
    coefs_mf = encoder_mf.encoder.coef_

    tv_idxs = [
        i for i, dm_name in enumerate(encoder_mb.dm_names) if dm_name in regressors
    ]

    coefs_mb_ = coefs_mb[:, tv_idxs]
    coefs_mf_ = coefs_mf[:, tv_idxs]

    coefs["mb"]["DLS"].extend(coefs_mb_[encoder_mb.reg_idxs["DLS"]])
    coefs["mb"]["DMS"].extend(coefs_mb_[encoder_mb.reg_idxs["DMS"]])
    coefs["mf"]["DLS"].extend(coefs_mf_[encoder_mb.reg_idxs["DLS"]])
    coefs["mf"]["DMS"].extend(coefs_mf_[encoder_mb.reg_idxs["DMS"]])

coefs = {
    strategy: {region: np.array(coefs[strategy][region]) for region in coefs[strategy]}
    for strategy in coefs
}

In [ ]:
region = "DLS"
regr = "response"
val = "right"


def plot_bweight_strategy(reg, regr, val):
    regr_idx = np.where(regressors == f"{regr}_{val}")[0]

    plt.figure(figsize=(2.5, 2), tight_layout=True)
    plt.scatter(
        coefs["mb"][reg][:, regr_idx],
        coefs["mf"][reg][:, regr_idx],
        s=0.5,
        alpha=0.5,
        zorder=2,
    )

    plt.plot([-1.5, 1.5], [-1.5, 1.5], linewidth=0.5, linestyle="--", color="#666666")
    plt.axhline(y=0, color="k", linewidth=0.5)
    plt.axvline(x=0, color="k", linewidth=0.5)

    plt.xlabel(rf"mb $\beta$ {regr}_{val}")
    plt.ylabel(rf"mf $\beta$ {regr}_{val}")

    plt.show()


plot_bweight_strategy(reg="DLS", regr="response", val="left")